# Muon Step-Length Hit Energy Comparison

Compare `SiPadHits.energy` distributions for the 100 GeV muon samples generated with different Geant4 step-length settings. Hit energy is converted from GeV to MeV.

## Setup

In [1]:
from __future__ import annotations

import re
from pathlib import Path

import ROOT

ROOT.gStyle.SetOptStat(0)
ROOT.EnableImplicitMT()


## Input Samples

In [2]:
sample_dir = Path("/home/llr/ilc/shi/data/siwecal_k4sim/output/muon")
tree_name = "events"
collection = "SiPadHits"

step_pattern = re.compile(r"mu-_100GeV_(?P<step>[0-9]+(?:um|mm)Step)\.edm4hep\.root")


def step_to_mm(step_name: str) -> float:
    value = float(re.match(r"[0-9]+", step_name).group(0))
    if "um" in step_name:
        return value / 1000.0
    return value


def step_label(step_name: str) -> str:
    step_mm = step_to_mm(step_name)
    return f"{step_mm:g} mm step"


step_files = []
for path in sorted(sample_dir.glob("mu-_100GeV_*Step.edm4hep.root")):
    match = step_pattern.fullmatch(path.name)
    if match:
        step_name = match.group("step")
        step_files.append((step_to_mm(step_name), step_label(step_name), path))

if not step_files:
    raise FileNotFoundError(f"No step samples found under {sample_dir}")

samples = {label: path for _, label, path in sorted(step_files)}
samples


{'0.25 mm step': PosixPath('/home/llr/ilc/shi/data/siwecal_k4sim/output/muon/mu-_100GeV_250umStep.edm4hep.root'),
 '0.5 mm step': PosixPath('/home/llr/ilc/shi/data/siwecal_k4sim/output/muon/mu-_100GeV_500umStep.edm4hep.root'),
 '1 mm step': PosixPath('/home/llr/ilc/shi/data/siwecal_k4sim/output/muon/mu-_100GeV_1mmStep.edm4hep.root')}

## Load Hits

In [3]:
energy_leaf = f"{collection}.energy"


def make_hit_frame(path: Path):
    return (
        ROOT.RDataFrame(tree_name, str(path))
        .Alias("hit_energy", energy_leaf)
        .Define("hit_energy_mev", "1000.0f * hit_energy")
    )


frames = {label: make_hit_frame(path) for label, path in samples.items()}
frames


{'0.25 mm step': <cppyy.gbl.ROOT.RDF.RInterface<ROOT::Detail::RDF::RLoopManager,void> object at 0xbcefb60>,
 '0.5 mm step': <cppyy.gbl.ROOT.RDF.RInterface<ROOT::Detail::RDF::RLoopManager,void> object at 0xbccc570>,
 '1 mm step': <cppyy.gbl.ROOT.RDF.RInterface<ROOT::Detail::RDF::RLoopManager,void> object at 0xc910990>}

In [4]:
def summarize(label: str, path: Path, df):
    n_events = int(ROOT.RDataFrame(tree_name, str(path)).Count().GetValue())
    n_hits = int(df.Define("hit_count", "hit_energy.size()").Sum("hit_count").GetValue())
    return {
        "sample": label,
        "input_file": str(path),
        "events": n_events,
        "SiPadHits": n_hits,
        "mean_hit_energy_MeV": float(df.Mean("hit_energy_mev").GetValue()) if n_hits else 0.0,
        "max_hit_energy_MeV": float(df.Max("hit_energy_mev").GetValue()) if n_hits else 0.0,
        "total_hit_energy_MeV": float(df.Sum("hit_energy_mev").GetValue()) if n_hits else 0.0,
    }


summaries = {
    label: summarize(label, samples[label], frames[label])
    for label in samples
}
summaries


{'0.25 mm step': {'sample': '0.25 mm step',
  'input_file': '/home/llr/ilc/shi/data/siwecal_k4sim/output/muon/mu-_100GeV_250umStep.edm4hep.root',
  'events': 1000,
  'SiPadHits': 19035,
  'mean_hit_energy_MeV': 0.30658903170915913,
  'max_hit_energy_MeV': 57.77655029296875,
  'total_hit_energy_MeV': 5835.922218583844},
 '0.5 mm step': {'sample': '0.5 mm step',
  'input_file': '/home/llr/ilc/shi/data/siwecal_k4sim/output/muon/mu-_100GeV_500umStep.edm4hep.root',
  'events': 1000,
  'SiPadHits': 18335,
  'mean_hit_energy_MeV': 0.27031479678013026,
  'max_hit_energy_MeV': 36.93924331665039,
  'total_hit_energy_MeV': 4956.221798963688},
 '1 mm step': {'sample': '1 mm step',
  'input_file': '/home/llr/ilc/shi/data/siwecal_k4sim/output/muon/mu-_100GeV_1mmStep.edm4hep.root',
  'events': 1000,
  'SiPadHits': 18043,
  'mean_hit_energy_MeV': 0.26336013779380046,
  'max_hit_energy_MeV': 14.437545776367188,
  'total_hit_energy_MeV': 4751.806966213542}}

## Hit Energy Distribution

In [5]:
energy_range_mev = (0.0, 2.0)
energy_bins = 200
colors = [ROOT.kBlue + 1, ROOT.kRed + 1, ROOT.kGreen + 2, ROOT.kMagenta + 1, ROOT.kOrange + 7]

energy_hists = []
for index, (label, df) in enumerate(frames.items()):
    safe_label = re.sub(r"[^A-Za-z0-9_]", "_", label)
    hist = df.Histo1D(
        (
            f"h_hit_energy_mev_{safe_label}",
            "SiPadHits energy;Hit energy [MeV];Hits",
            energy_bins,
            energy_range_mev[0],
            energy_range_mev[1],
        ),
        "hit_energy_mev",
    )
    energy_hists.append((label, hist, colors[index % len(colors)]))

canvas_counts = ROOT.TCanvas("canvas_step_hit_energy_counts", "Step-length hit energy comparison", 850, 650)
legend_counts = ROOT.TLegend(0.60, 0.68, 0.88, 0.88)

max_count = max(float(hist.GetMaximum()) for _, hist, _ in energy_hists)
for index, (label, hist, color) in enumerate(energy_hists):
    hist.SetLineColor(color)
    hist.SetLineWidth(2)
    hist.SetMaximum(max_count * 1.15 if max_count > 0 else 1.0)
    hist.Draw("hist" if index == 0 else "hist same")
    legend_counts.AddEntry(hist.GetPtr(), label, "l")

legend_counts.Draw()
canvas_counts.Draw()


## Normalized Comparison

In [6]:
normalized_hists = []
for label, hist, color in energy_hists:
    clone = hist.GetPtr().Clone(f"{hist.GetName()}_normalized")
    integral = clone.Integral()
    if integral > 0:
        clone.Scale(1.0 / integral)
    clone.SetDirectory(0)
    normalized_hists.append((label, clone, color))

canvas_norm = ROOT.TCanvas("canvas_step_hit_energy_normalized", "Normalized step-length hit energy comparison", 850, 650)
legend_norm = ROOT.TLegend(0.60, 0.68, 0.88, 0.88)

max_norm = max(float(hist.GetMaximum()) for _, hist, _ in normalized_hists)
for index, (label, hist, color) in enumerate(normalized_hists):
    hist.SetTitle("SiPadHits energy, normalized;Hit energy [MeV];Fraction of hits")
    hist.SetLineColor(color)
    hist.SetLineWidth(2)
    hist.SetMaximum(max_norm * 1.15 if max_norm > 0 else 1.0)
    hist.Draw("hist" if index == 0 else "hist same")
    legend_norm.AddEntry(hist, label, "l")

legend_norm.Draw()
canvas_norm.Draw()


## Peak Values

In [7]:
energy_peaks = {}
for label, hist, _ in energy_hists:
    max_bin = hist.GetMaximumBin()
    energy_peaks[label] = {
        "peak_energy_MeV": float(hist.GetBinCenter(max_bin)),
        "peak_bin_count": int(hist.GetBinContent(max_bin)),
        "bin_width_MeV": float(hist.GetBinWidth(max_bin)),
    }

energy_peaks


{'0.25 mm step': {'peak_energy_MeV': 0.145,
  'peak_bin_count': 2054,
  'bin_width_MeV': 0.01},
 '0.5 mm step': {'peak_energy_MeV': 0.145,
  'peak_bin_count': 2130,
  'bin_width_MeV': 0.01},
 '1 mm step': {'peak_energy_MeV': 0.145,
  'peak_bin_count': 2022,
  'bin_width_MeV': 0.01}}